# 08 — DAP + Few-Shot Fine-Tuning (Arm 2)



In [1]:

!rm -rf /content/drive/MyDrive/AAI590/data/processed/results/dap_fewshot
!pip -q install "transformers==4.44.2" "datasets==2.19.2" "seqeval==1.2.2" "accelerate>=0.26.0" pandas matplotlib
import torch
print('GPU available:', torch.cuda.is_available())

GPU available: True


## Quiet mode

The cells below intentionally trigger a few harmless warnings from `transformers` and `huggingface_hub` (e.g. a note that no `HF_TOKEN` is set, or that a randomly-initialized head is expected). This cell silences those specific, known-benign warnings so the notebook output stays readable, without hiding real errors.

In [2]:
import os
import warnings
import logging

# Only suppress the specific noisy-but-harmless categories we actually expect --
# real errors (exceptions) are never affected by any of this.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
warnings.filterwarnings("ignore", message=".*newly initialized.*")

os.environ["TOKENIZERS_PARALLELISM"] = "false"      # silences the tokenizer fork warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"  # silences the Windows-symlink notice

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # transformers: only show actual errors, not routine notices
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


## Step 1 — Mount Drive and configure

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

# --- executed-pipeline layout  ---
PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
TOKENIZED_DIR  = PROCESSED / 'tokenized'         # HF datasets saved by NB03
LABELS_DIR     = PROCESSED / 'label_maps'        # <ds>_label_map.json (NB03)
MODELS_DIR     = PROCESSED / 'models'            # baseline_conll2003 (NB04)
RESULTS_DIR    = PROCESSED / 'results'
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'    # raw jsonl demos (NB02)
BASELINE_DIR   = MODELS_DIR / 'baseline_conll2003'

# shared experiment grid (identical to NB05/NB06)
TARGET_DATASETS = ['wnut17', 'scierc']
BUDGETS = [50, 100, 200]
SEEDS   = [13, 42, 101]

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_label_map(dataset_name):
    with open(LABELS_DIR / f'{dataset_name}_label_map.json') as f:
        m = json.load(f)
    label2id = {str(k): int(v) for k, v in m['label2id'].items()}
    id2label = {int(k): str(v) for k, v in m['id2label'].items()}
    return label2id, id2label

print('processed dir :', PROCESSED)
print('baseline dir  :', BASELINE_DIR, '(exists:', BASELINE_DIR.exists(), ')')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
processed dir : /content/drive/MyDrive/AAI590/data/processed
baseline dir  : /content/drive/MyDrive/AAI590/data/processed/models/baseline_conll2003 (exists: True )


In [4]:
import gc, random
import numpy as np
import pandas as pd
from datasets import load_from_disk
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from transformers import (AutoConfig, AutoModelForMaskedLM, AutoModelForTokenClassification,
                          AutoTokenizer, DataCollatorForTokenClassification,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)


LEARNING_RATE = 2e-5
# Budget-scaled epoch counts
NUM_EPOCHS_BY_BUDGET = {50: 20, 100: 12, 200: 8}
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2
SAVE_ALL_MODELS = False

DAP_FEWSHOT_RESULTS_DIR = RESULTS_DIR / 'dap_fewshot'
DAP_FEWSHOT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for ds in TARGET_DATASETS:
    assert (MODELS_DIR / f'dap_{ds}').exists(), f'Run Notebook 07 first (models/dap_{ds} missing).'

## Step 2 — Cache each DAP-adapted encoder

We transplant the DAP encoder into a fresh `bert-base-cased` token-classification model with a
newly-initialized head sized to the target label set — exactly NB05's construction, but the
encoder source is the DAP checkpoint instead of the CoNLL baseline.

In [5]:
dap_encoder_state = {}
for ds_name in TARGET_DATASETS:
    mlm = AutoModelForMaskedLM.from_pretrained(str(MODELS_DIR / f'dap_{ds_name}'))
    dap_encoder_state[ds_name] = {k: v.clone() for k, v in mlm.base_model.state_dict().items()}
    del mlm
print('cached DAP encoders for:', list(dap_encoder_state))

tokenizer = AutoTokenizer.from_pretrained(str(BASELINE_DIR), clean_up_tokenization_spaces=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

cached DAP encoders for: ['wnut17', 'scierc']


## Step 3 — Helper functions (identical metrics to Notebook 05)

In [6]:
def collapse_to_boundary(tag):
    if tag == 'O':
        return 'O'
    if tag.startswith('B-'):
        return 'B-ENT'
    if tag.startswith('I-'):
        return 'I-ENT'
    raise ValueError(f'Unexpected BIO tag: {tag}')

def decode_predictions(eval_pred, id2label):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_tags, pred_tags = [], []
    for pred_row, label_row in zip(predictions, labels):
        st, sp = [], []
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue
            st.append(id2label[int(l)]); sp.append(id2label[int(p)])
        true_tags.append(st); pred_tags.append(sp)
    return true_tags, pred_tags

def build_compute_metrics(id2label):
    def compute_metrics(eval_pred):
        true_tags, pred_tags = decode_predictions(eval_pred, id2label)
        tb = [[collapse_to_boundary(t) for t in s] for s in true_tags]
        pb = [[collapse_to_boundary(t) for t in s] for s in pred_tags]
        return {
            'typed_precision': precision_score(true_tags, pred_tags),
            'typed_recall':    recall_score(true_tags, pred_tags),
            'typed_f1':        f1_score(true_tags, pred_tags),
            'boundary_precision': precision_score(tb, pb),
            'boundary_recall':    recall_score(tb, pb),
            'boundary_f1':        f1_score(tb, pb),
        }
    return compute_metrics

def per_type_f1(trainer, dataset, id2label):
    preds, labels, _ = trainer.predict(dataset)
    preds = np.argmax(preds, axis=2)
    true_tags, pred_tags = [], []
    for pr, lr in zip(preds, labels):
        st, sp = [], []
        for p, l in zip(pr, lr):
            if l == -100:
                continue
            st.append(id2label[int(l)]); sp.append(id2label[int(p)])
        true_tags.append(st); pred_tags.append(sp)
    rep = classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)
    out = {}
    for k, v in rep.items():
        if k in ('micro avg', 'macro avg', 'weighted avg'):
            continue
        out[k] = {'precision': float(v['precision']), 'recall': float(v['recall']),
                  'f1': float(v['f1-score']), 'support': int(v['support'])}
    return out

def load_target_datasets(dataset_name, budget, seed):
    fewshot = TOKENIZED_DIR / 'fewshot' / dataset_name / f'budget{budget}_seed{seed}'
    val = TOKENIZED_DIR / dataset_name / 'validation'
    test = TOKENIZED_DIR / dataset_name / 'test'
    for p in (fewshot, val, test):
        assert p.exists(), f'Missing tokenized dataset: {p} (re-run Notebook 03).'
    return load_from_disk(str(fewshot)), load_from_disk(str(val)), load_from_disk(str(test))

def create_target_model(dataset_name):
    label2id, id2label = load_label_map(dataset_name)
    config = AutoConfig.from_pretrained('bert-base-cased', num_labels=len(id2label),
                                        label2id=label2id, id2label=id2label)
    model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', config=config,
                                                            ignore_mismatched_sizes=True)
    # transplant the DAP-adapted encoder (the ONE difference from NB05)
    model.base_model.load_state_dict(dap_encoder_state[dataset_name], strict=True)
    return model, label2id, id2label

def cleanup(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Step 4 — Run one DAP+few-shot experiment

In [7]:
import time

def run_experiment(dataset_name, budget, seed):
    print('=' * 80)
    print(f'[DAP+FS] Dataset={dataset_name} | Budget={budget} | Seed={seed} | Epochs={NUM_EPOCHS_BY_BUDGET[budget]}')
    print('=' * 80)
    set_seed(seed); random.seed(seed); np.random.seed(seed)

    train_ds, val_ds, test_ds = load_target_datasets(dataset_name, budget, seed)
    model, label2id, id2label = create_target_model(dataset_name)
    num_epochs = NUM_EPOCHS_BY_BUDGET[budget]

    args = TrainingArguments(
        output_dir=f'/content/dapfs_checkpoints/{dataset_name}_b{budget}_s{seed}',
        eval_strategy='epoch', save_strategy='epoch', logging_strategy='epoch',
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=num_epochs, weight_decay=WEIGHT_DECAY,
        load_best_model_at_end=True, metric_for_best_model='eval_typed_f1',
        greater_is_better=True, save_total_limit=1, report_to='none',
        seed=seed, data_seed=seed,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=build_compute_metrics(id2label),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    train_output = trainer.train()
    train_seconds = time.time() - t0
    test_metrics = trainer.evaluate(test_ds, metric_key_prefix='test')
    per_type = per_type_f1(trainer, test_ds, id2label)

    if SAVE_ALL_MODELS:
        d = MODELS_DIR / 'dap_fewshot' / dataset_name / f'budget{budget}_seed{seed}'
        d.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(d)); tokenizer.save_pretrained(str(d))

    result = {
        'method': 'dap_fewshot', 'dataset': dataset_name, 'budget': budget, 'seed': seed,
        'num_train_sentences': len(train_ds),
        'num_epochs': num_epochs,
        'training_loss': train_output.training_loss,
        'train_seconds': round(train_seconds, 1),
        'test_typed_precision': test_metrics.get('test_typed_precision'),
        'test_typed_recall':    test_metrics.get('test_typed_recall'),
        'test_typed_f1':        test_metrics.get('test_typed_f1'),
        'test_boundary_precision': test_metrics.get('test_boundary_precision'),
        'test_boundary_recall':    test_metrics.get('test_boundary_recall'),
        'test_boundary_f1':        test_metrics.get('test_boundary_f1'),
        'per_type': json.dumps(per_type),
    }
    print(f"  Typed F1: {result['test_typed_f1']:.4f} | Boundary F1: {result['test_boundary_f1']:.4f}")
    cleanup(trainer, model, train_ds, val_ds, test_ds)
    return result

## Step 5 — Run the full grid (18 runs, resumable)

In [8]:
RESULTS_CSV = DAP_FEWSHOT_RESULTS_DIR / 'dap_fewshot_results.csv'
RESULTS_JSON = DAP_FEWSHOT_RESULTS_DIR / 'dap_fewshot_results.json'

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    completed = {(r.dataset, int(r.budget), int(r.seed)) for r in results_df.itertuples()}
    all_results = results_df.to_dict('records')
    print(f'Resuming from {len(all_results)} completed runs.')
else:
    completed, all_results = set(), []

for ds_name in TARGET_DATASETS:
    for budget in BUDGETS:
        for seed in SEEDS:
            if (ds_name, budget, seed) in completed:
                print('Skipping', (ds_name, budget, seed)); continue
            all_results.append(run_experiment(ds_name, budget, seed))
            completed.add((ds_name, budget, seed))
            results_df = pd.DataFrame(all_results).sort_values(['dataset', 'budget', 'seed'])
            results_df.to_csv(RESULTS_CSV, index=False)
            with open(RESULTS_JSON, 'w') as f:
                json.dump(results_df.replace({np.nan: None}).to_dict('records'), f, indent=2, default=str)

results_df = pd.DataFrame(all_results).sort_values(['dataset', 'budget', 'seed']).reset_index(drop=True)
display(results_df[['dataset', 'budget', 'seed', 'test_typed_f1', 'test_boundary_f1']].round(4))
print('Saved ->', RESULTS_CSV)

[DAP+FS] Dataset=wnut17 | Budget=50 | Seed=13 | Epochs=20


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'loss': 1.6078, 'grad_norm': 7.614565849304199, 'learning_rate': 1.9e-05, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.4947810471057892, 'eval_typed_precision': 0.0, 'eval_typed_recall': 0.0, 'eval_typed_f1': 0.0, 'eval_boundary_precision': 0.0, 'eval_boundary_recall': 0.0, 'eval_boundary_f1': 0.0, 'eval_runtime': 3.3226, 'eval_samples_per_second': 303.681, 'eval_steps_per_second': 9.631, 'epoch': 1.0}
{'loss': 0.522, 'grad_norm': 4.780792713165283, 'learning_rate': 1.8e-05, 'epoch': 2.0}
{'eval_loss': 0.4167858064174652, 'eval_typed_precision': 0.0, 'eval_typed_recall': 0.0, 'eval_typed_f1': 0.0, 'eval_boundary_precision': 0.0, 'eval_boundary_recall': 0.0, 'eval_boundary_f1': 0.0, 'eval_runtime': 3.0298, 'eval_samples_per_second': 333.02, 'eval_steps_per_second': 10.562, 'epoch': 2.0}
{'loss': 0.3932, 'grad_norm': 2.4533936977386475, 'learning_rate': 1.7e-05, 'epoch': 3.0}
{'eval_loss': 0.32381248474121094, 'eval_typed_precision': 0.5760869565217391, 'eval_typed_recall': 0.12679425837320574, 'eval_typed_f1': 0.20784313725490194, 'eval_boundary_precision': 0.6666666666666666, 'eval_boun

...

{'eval_loss': 1.2732391357421875, 'eval_typed_precision': 0.0, 'eval_typed_recall': 0.0, 'eval_typed_f1': 0.0, 'eval_boundary_precision': 0.0, 'eval_boundary_recall': 0.0, 'eval_boundary_f1': 0.0, 'eval_runtime': 1.2405, 'eval_samples_per_second': 221.677, 'eval_steps_per_second': 7.255, 'epoch': 1.0}
{'loss': 1.0205, 'grad_norm': 6.729172229766846, 'learning_rate': 1.8e-05, 'epoch': 2.0}
{'eval_loss': 0.9750571250915527, 'eval_typed_precision': 0.005128205128205128, 'eval_typed_recall': 0.0012330456226880395, 'eval_typed_f1': 0.0019880715705765406, 'eval_boundary_precision': 0.0582010582010582, 'eval_boundary_recall': 0.013563501849568433, 'eval_boundary_f1': 0.022, 'eval_runtime': 1.2504, 'eval_samples_per_second': 219.925, 'eval_steps_per_second': 7.198, 'epoch': 2.0}
{'loss': 0.8148, 'grad_norm': 3.2207534313201904, 'learning_rate': 1.7e-05, 'epoch': 3.0}
{'eval_loss': 0.9042948484420776, 'eval_typed_precision': 0.02218430034129693, 'eval_typed_recall': 0.016029593094944512, 'eval_

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.9894567728042603, 'eval_typed_precision': 0.0, 'eval_typed_recall': 0.0, 'eval_typed_f1': 0.0, 'eval_boundary_precision': 0.0, 'eval_boundary_recall': 0.0, 'eval_boundary_f1': 0.0, 'eval_runtime': 1.2453, 'eval_samples_per_second': 220.822, 'eval_steps_per_second': 7.227, 'epoch': 1.0}
{'loss': 0.8465, 'grad_norm': 4.316721439361572, 'learning_rate': 1.6666666666666667e-05, 'epoch': 2.0}
{'eval_loss': 0.9137762784957886, 'eval_typed_precision': 0.04609218436873747, 'eval_typed_recall': 0.02836004932182491, 'eval_typed_f1': 0.035114503816793895, 'eval_boundary_precision': 0.15, 'eval_boundary_recall': 0.07768187422934648, 'eval_boundary_f1': 0.10235580828594638, 'eval_runtime': 1.2302, 'eval_samples_per_second': 223.535, 'eval_steps_per_second': 7.316, 'epoch': 2.0}
{'loss': 0.7048, 'grad_norm': 3.166910409927368, 'learning_rate': 1.5000000000000002e-05, 'epoch': 3.0}
{'eval_loss': 0.8164066672325134, 'eval_typed_precision': 0.07870837537840565, 'eval_typed_recall': 0.09

,dataset,budget,seed,test_typed_f1,test_boundary_f1
0,scierc,50,13,0.2675,0.5258
1,scierc,50,42,0.2470,0.5401
2,scierc,50,101,0.2449,0.5419
3,scierc,100,13,0.3234,0.6027
4,scierc,100,42,0.2955,0.5975
5,scierc,100,101,0.2807,0.5988
6,scierc,200,13,0.3879,0.6396
7,scierc,200,42,0.3355,0.6226
8,scierc,200,101,0.3614,0.6419
9,wnut17,50,13,0.3190,0.4835


Saved -> /content/drive/MyDrive/AAI590/data/processed/results/dap_fewshot/dap_fewshot_results.csv


## Step 6 — Aggregate across seeds + compare Arm 1 vs Arm 2

In [9]:
metric_cols = ['test_typed_f1', 'test_boundary_f1']
summary = (results_df.groupby(['dataset', 'budget'])[metric_cols]
           .agg(['mean', 'std']).reset_index())
summary.columns = ['_'.join(str(p) for p in c if str(p)) if isinstance(c, tuple) else c
                   for c in summary.columns]
summary.to_csv(DAP_FEWSHOT_RESULTS_DIR / 'dap_fewshot_summary_by_budget.csv', index=False)
display(summary.round(4))

# side-by-side with Notebook 05 (Arm 1), if present
ARM1_CSV = RESULTS_DIR / 'fewshot_v2' / 'fewshot_results.csv'  # fixed budget-scaled Arm 1 output
if ARM1_CSV.exists():
    a1 = pd.read_csv(ARM1_CSV)
    a1g = a1.groupby(['dataset', 'budget'])['test_typed_f1'].mean().reset_index()
    a1g = a1g.rename(columns={'test_typed_f1': 'arm1_typed_f1'})
    a2g = results_df.groupby(['dataset', 'budget'])['test_typed_f1'].mean().reset_index()
    a2g = a2g.rename(columns={'test_typed_f1': 'arm2_typed_f1'})
    cmp = a1g.merge(a2g, on=['dataset', 'budget'])
    cmp['dap_gain'] = cmp['arm2_typed_f1'] - cmp['arm1_typed_f1']
    cmp.to_csv(DAP_FEWSHOT_RESULTS_DIR / 'arm1_vs_arm2_typed_f1.csv', index=False)
    print('\nArm 1 (few-shot) vs Arm 2 (DAP + few-shot) -- typed F1, mean over seeds:')
    display(cmp.round(4))
else:
    print('Notebook 05 results not found -- run NB05 for the Arm 1 vs Arm 2 comparison.')

,dataset,budget,test_typed_f1_mean,test_typed_f1_std,test_boundary_f1_mean,test_boundary_f1_std
0,scierc,50,0.2531,0.0125,0.5359,0.0088
1,scierc,100,0.2999,0.0217,0.5997,0.0027
2,scierc,200,0.3616,0.0262,0.6347,0.0106
3,wnut17,50,0.2201,0.1909,0.3350,0.2907
4,wnut17,100,0.3575,0.0042,0.5582,0.0098
5,wnut17,200,0.3823,0.0087,0.5908,0.0180



Arm 1 (few-shot) vs Arm 2 (DAP + few-shot) -- typed F1, mean over seeds:


,dataset,budget,arm1_typed_f1,arm2_typed_f1,dap_gain
0,scierc,50,0.2391,0.2531,0.0141
1,scierc,100,0.3081,0.2999,-0.0082
2,scierc,200,0.3732,0.3616,-0.0116
3,wnut17,50,0.3116,0.2201,-0.0915
4,wnut17,100,0.3787,0.3575,-0.0212
5,wnut17,200,0.4027,0.3823,-0.0205
